In [1]:
!pip install gym numpy matplotlib imageio


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import numpy as np
import gym
import random
import matplotlib.pyplot as plt
import imageio

np.bool8 = np.bool_  #NumPy ' ın yeni sürümlerindeki Gym uyuşmazlıklarını çözmek için

In [ ]:
# 1. STANDART TAXI-V3 EĞİTİMİ (5x5, 4 Hedef)

def train_gym_v3():
    print("1. Standart Gym Taxi-v3 Eğitimi Başlıyor...")
    # Yeni gym versiyonlarındaki render hatalarını önlemek için render_mode belirtiyoruz
    try:
        env = gym.make("Taxi-v3", render_mode="ansi")
    except:
        env = gym.make("Taxi-v3") # Eski gym versiyonları için
        
    q_table = np.zeros([env.observation_space.n, env.action_space.n]) # q table'ı sıfırlarla dolu sıfır matrisi olarak tanımladık. satır durum, sütun hareketler.
    alpha, gamma, epsilon = 0.1, 0.6, 0.1  #öğrenme oranı, ödül önemi, keşif oranı. bizim belirleyeceğimiz hiperparametreler.
    episodes = 5000 #toplam oyun sayısı

    for i in range(episodes):
        state = env.reset() #her oyunda environment'ı sıfırla.
        if isinstance(state, tuple): state = state[0] # Gym v0.26+ uyumluluğu
        done = False
        while not done:  #yolcu hedefe gidene kadar döngüye devam
            if random.uniform(0, 1) < epsilon:
                action = env.action_space.sample() #%10 ihtimalle random hareket et.
            else:
                action = np.argmax(q_table[state]) #%90 ihtimalle q tablosundaki en mantıklı hareketi yap.
            
            #hareket durumu ve ortamın bize geri dönüş parametreleri
            step_res = env.step(action)
            if len(step_res) == 5: # Gym v0.26+
                next_state, reward, terminated, truncated, _ = step_res
                done = terminated or truncated
            else:
                next_state, reward, done, _ = step_res

            # q table güncelleme bellman denklemine göre.   
            old_value = q_table[state, action] 
            next_max = np.max(q_table[next_state])
            q_table[state, action] = (1 - alpha) * old_value + alpha * (reward + gamma * next_max)
            state = next_state  #taksinin yeni konumunu güncelleme.
    print("Standart Taxi-v3 Eğitimi Tamamlandı!\n")

In [ ]:
# 2. ÖZEL 6x6 SMARTCAB ORTAMI (6x6, 5 Hedef)

class Taxi6x6Env:
    def __init__(self):
        self.num_rows = 6
        self.num_columns = 6
        # 5 Hedef Noktası: R, G, Y, B ve yeni eklenen M(5,5)
        self.locs = [(0,0), (0,4), (4,0), (4,3), (5,5)] 
        
        # Durum Sayısı = 6 (satır) * 6 (sütun) * 6 (5 durak + 1 taksi içi) * 5 (hedef) = 1080
        self.n_states = 1080 
        self.n_actions = 6 # Güney, Kuzey, Doğu, Batı, Al, Bırak
        
        #Duvarların koordinatları
        self.right_walls = [(0,1), (1,1), (3,0), (4,0), (3,2), (4,2)] 
        
        # Geçiş matrisi (State -> Action -> Sonuç)
        self.P = {state: {action: [] for action in range(self.n_actions)} for state in range(self.n_states)}
        self._build_transitions()
        self.s = self.reset()

    #q tablosu tek boyutludur. bizim x,y koordinatları, yolcu durumu ve hedef yeri gibi
    #farklı durumumuz vardır. bu 4 bilgiyi tek bir id'ye sıkıştırıp boyut indirgeyecez.
    def encode(self, row, col, pass_idx, dest_idx):
        i = row
        i *= 6; i += col
        i *= 6; i += pass_idx
        i *= 5; i += dest_idx
        return i

    #decode kısmında geri çözüyoruz encode kısmını.agent q table'dan bilgiyi bu şekilde alacak.
    def decode(self, i):
        out = []
        out.append(i % 5); i = i // 5
        out.append(i % 6); i = i // 6
        out.append(i % 6); i = i // 6
        out.append(i)
        return list(reversed(out))

    #Tüm ihtimallerin döngüsü. duvara çarparsan gidemezsin, ceza ödül sistemi vs.
    def _build_transitions(self):
        for row in range(self.num_rows):
            for col in range(self.num_columns):
                for pass_idx in range(6): 
                    for dest_idx in range(5):
                        state = self.encode(row, col, pass_idx, dest_idx)
                        for action in range(6):
                            new_row, new_col, new_pass_idx = row, col, pass_idx
                            reward = -1
                            done = False
                            taxi_loc = (row, col)
                            
                            if action == 0: # Güney
                                new_row = min(row + 1, self.num_rows - 1)
                            elif action == 1: # Kuzey
                                new_row = max(row - 1, 0)
                            elif action == 2: # Doğu
                                if (row, col) not in self.right_walls and col < self.num_columns - 1:
                                    new_col += 1
                            elif action == 3: # Batı
                                if (row, col - 1) not in self.right_walls and col > 0:
                                    new_col -= 1
                            #ödül ve ceza sistemimiz
                            elif action == 4: # Yolcu Al
                                if pass_idx < 5 and taxi_loc == self.locs[pass_idx]:
                                    new_pass_idx = 5 #yolcu taksiye bindi.
                                else:
                                    reward = -10 #yolcu yokken kapıyı açarsan ceza.
                            elif action == 5: # Yolcu İndir
                                if pass_idx == 5 and taxi_loc == self.locs[dest_idx]:
                                    new_pass_idx = dest_idx
                                    done = True #oyun başarı ile bitti.
                                    reward = 20 #büyük ödül.
                                elif taxi_loc in self.locs and pass_idx == 5:
                                    reward = -10 # Yanlış durağa bırakma
                                else:
                                    reward = -10
                                    
                            new_state = self.encode(new_row, new_col, new_pass_idx, dest_idx)
                            self.P[state][action].append((1.0, new_state, reward, done))

    def reset(self):
        state = random.randint(0, self.n_states - 1)
        row, col, pass_idx, dest_idx = self.decode(state)
        # Yolcu hedefe zaten varmış olmasın ve oyun başlarken taksinin içinde olmasın
        while pass_idx == dest_idx or pass_idx == 5:
            state = random.randint(0, self.n_states - 1)
            row, col, pass_idx, dest_idx = self.decode(state)
        self.s = state
        return self.s

    def step(self, action):
        transitions = self.P[self.s][action]
        prob, next_state, reward, done = transitions[0]
        self.s = next_state
        return next_state, reward, done, {}

In [ ]:
# 3. YENİ ORTAMIN EĞİTİLMESİ VE GRAFİK

def train_custom_6x6(env):
    print("--- 2. Özel 6x6 Ortam Eğitimi Başlıyor ---")
    q_table = np.zeros([env.n_states, env.n_actions])
    alpha, gamma, epsilon = 0.1, 0.6, 0.1
    episodes = 10000
    
    rewards_history = []
    
    for i in range(1, episodes + 1):
        state = env.reset()
        total_reward = 0
        done = False
        
        while not done:
            if random.uniform(0, 1) < epsilon:
                action = random.randint(0, 5)
            else:
                action = np.argmax(q_table[state])
            
            next_state, reward, done, _ = env.step(action)
            
            old_value = q_table[state, action]
            next_max = np.max(q_table[next_state])
            q_table[state, action] = (1 - alpha) * old_value + alpha * (reward + gamma * next_max)
            
            state = next_state
            total_reward += reward
            
        rewards_history.append(total_reward) #grafik çizimi için her oyunun toplam ödülünü hafızada tutuyoruz.
        if i % 2000 == 0:
            print(f"Episode {i}/{episodes} tamamlandı.")
            
    # Grafiği Çiz (Hareketli Ortalama)
    plt.figure(figsize=(10, 5))
    window = 100
    smoothed_rewards = [np.mean(rewards_history[max(0, i-window):i+1]) for i in range(len(rewards_history))]
    plt.plot(smoothed_rewards, color='blue')
    plt.title('6x6 SmartCab Egitim Sureci (Odul Gelisimi)')
    plt.xlabel('Episode')
    plt.ylabel('Ortalama Odul (100 Epoch)')
    plt.grid(True)
    plt.savefig('toplam_odul.jpg')
    plt.close()
    print("Eğitim Grafiği 'toplam_odul.jpg' olarak kaydedildi.")
    
    np.save('q_table.npy', q_table)
    print("Q-Table 'q_table.npy' olarak kaydedildi.\n")
    return q_table

In [ ]:
# 4. TEST VE GIF OLUŞTURMA

def draw_state(env, state):
    row, col, pass_idx, dest_idx = env.decode(state)
    fig, ax = plt.subplots(figsize=(5,5))
    ax.set_xlim(0, 6)
    ax.set_ylim(0, 6)
    ax.invert_yaxis()
    
    # Izgarayı Çiz
    for i in range(7):
        ax.axhline(i, color='black', lw=1)
        ax.axvline(i, color='black', lw=1)
        
    # Duvarları Çiz
    for (r, c) in env.right_walls:
        ax.plot([c+1, c+1], [r, r+1], color='black', lw=5)
        
    # Durakları Renklendir
    colors = ['red', 'green', 'yellow', 'blue', 'magenta']
    for i, loc in enumerate(env.locs):
        rect = plt.Rectangle((loc[1], loc[0]), 1, 1, color=colors[i], alpha=0.3)
        ax.add_patch(rect)
        
    # Yolcu ve Hedef Metinleri
    if pass_idx < 5:
        p_loc = env.locs[pass_idx]
        ax.text(p_loc[1]+0.5, p_loc[0]+0.5, 'Yolcu', fontsize=10, ha='center', va='center', weight='bold')
        
    d_loc = env.locs[dest_idx]
    ax.text(d_loc[1]+0.5, d_loc[0]+0.5, 'Hedef', fontsize=10, ha='center', va='center', weight='bold')
    
    # Taksi Çizimi (Yolcu bindiyse Yeşil, boşsa Turuncu)
    taxi_color = 'limegreen' if pass_idx == 5 else 'orange'
    rect = plt.Rectangle((col, row), 1, 1, color=taxi_color, alpha=0.9)
    ax.add_patch(rect)
    
    ax.axis('off')
    
    # Matplotlib figürünü görüntüye (numpy array) çevir
    fig.canvas.draw()
    try:
        image = np.array(fig.canvas.buffer_rgba())[..., :3]
    except AttributeError: # Eski sürümler için fallback
        image = np.frombuffer(fig.canvas.tostring_rgb(), dtype='uint8')
        image = image.reshape(fig.canvas.get_width_height()[::-1] + (3,))
    plt.close(fig)
    return image

#modelin test edilmesi ve gif oluşturma
def create_gif(env, q_table):
    print("--- 3. Ajan Test Ediliyor ve GIF Oluşturuluyor ---")
    state = env.reset()
    frames = []
    done = False
    steps = 0
    
    frames.append(draw_state(env, state))
    
    while not done and steps < 30: # Sonsuz döngüyü engellemek için max 30 adım
        action = np.argmax(q_table[state])
        state, reward, done, _ = env.step(action)
        frames.append(draw_state(env, state))
        steps += 1
        
    # GIF'i kaydet (fps=3 saniyede 3 kare oynatır)
    imageio.mimsave('taksi_6x6.gif', frames, fps=3)
    print(f"GIF 'taksi_6x6.gif' olarak başarıyla kaydedildi! (Toplam Adım: {steps})")


# ANA ÇALIŞTIRMA BLOĞU

if __name__ == "__main__":
    # 1. Standart ortamı çalıştır ve eğit
    train_gym_v3()
    
    # 2. 6x6 Özel Ortamı başlat
    env_6x6 = Taxi6x6Env()
    
    # 3. 6x6 Ortamı Eğit ve Sonuçları Kaydet
    q_table_6x6 = train_custom_6x6(env_6x6)
    
    # 4. Ajanı izle ve GIF çıkar
    create_gif(env_6x6, q_table_6x6)
    
    print("\nTebrikler! Tüm süreç tamamlandı. Dosyalarını (GIF, Grafik ve NPY) kontrol edebilirsin.")

--- 1. Standart Gym Taxi-v3 Eğitimi Başlıyor ---
Standart Taxi-v3 Eğitimi Tamamlandı!

--- 2. Özel 6x6 Ortam Eğitimi Başlıyor ---
Episode 2000/10000 tamamlandı.
Episode 4000/10000 tamamlandı.
Episode 6000/10000 tamamlandı.
Episode 8000/10000 tamamlandı.
Episode 10000/10000 tamamlandı.
Eğitim Grafiği 'toplam_odul.jpg' olarak kaydedildi.
Q-Table 'q_table.npy' olarak kaydedildi.

--- 3. Ajan Test Ediliyor ve GIF Oluşturuluyor ---
GIF 'taksi_6x6_6x6.gif' olarak başarıyla kaydedildi! (Toplam Adım: 10)

Tebrikler! Tüm süreç tamamlandı. Dosyalarını (GIF, Grafik ve NPY) kontrol edebilirsin.
